# LangGraph basics with toy graphs

Five tiny graphs, one concept each, then a sixth that combines them, using the same LangGraph features the incident-triage project uses. No Azure, no LLM, nothing from the project: just the ideas.

| # | Concept | Office picture | Used in the project for |
|---|---|---|---|
| 1 | State, nodes, edges | Whiteboard, workers, arrows | The whole agent flow |
| 2 | Parallel and reducers | Three people work at once, notes are merged | Logs, Metrics, Change agents |
| 3 | Conditional edges | A decision diamond | The Diagnosis loop |
| 4 | Interrupt and resume | "Stop until the manager signs" | The approval gate |
| 5 | Checkpointer and thread id | A save game | Pausing and resuming a run |
| 6 | All of it together | Four layers of workers | The whole triage graph, shaped like it |

**To run it:** Python 3.12 and `pip install langgraph`. Or from the repo root: `pip install -e "backend[dev]"`.

## 1. State, nodes and edges

- **State** is a dict every node can read. Here it has two keys.
- A **node** is a plain function. It receives the state and returns *only the keys it wants to change*.
- An **edge** says which node runs next. `START` and `END` mark the ends.

In [1]:
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    name: str
    greeting: str


def pick_greeting(state: State) -> dict:
    return {"greeting": f"hello {state['name']}"}      # returns only what it changes


def shout(state: State) -> dict:
    return {"greeting": state["greeting"].upper() + "!"}


g = StateGraph(State)
g.add_node("pick_greeting", pick_greeting)
g.add_node("shout", shout)
g.add_edge(START, "pick_greeting")
g.add_edge("pick_greeting", "shout")
g.add_edge("shout", END)
app = g.compile()

print(app.invoke({"name": "sam"}))      # the final state: input keys plus whatever the nodes added

{'name': 'sam', 'greeting': 'HELLO SAM!'}


## 2. Parallel nodes and reducers

Give one node several outgoing edges and they all run **at the same time**. A node with several incoming edges waits for **all** of them.

There is a catch: if three nodes write the *same key* at once, LangGraph needs to know how to combine them. That rule is a **reducer**. Without one, it refuses. This is why the project's `findings` key has a merge rule.

In [2]:
from langgraph.errors import InvalidUpdateError


def merge(a: dict, b: dict) -> dict:           # the reducer: combine two dicts
    return {**a, **b}


class Plain(TypedDict, total=False):
    n: int
    results: dict                              # no reducer
    total: int


class Merged(TypedDict, total=False):
    n: int
    results: Annotated[dict, merge]            # with a reducer
    total: int


def build(state_type):
    def double(s): return {"results": {"double": s["n"] * 2}}
    def square(s): return {"results": {"square": s["n"] ** 2}}
    def negate(s): return {"results": {"negate": -s["n"]}}
    def total(s):  return {"total": sum(s["results"].values())}

    g = StateGraph(state_type)
    for fn in (double, square, negate, total):
        g.add_node(fn.__name__, fn)
    for worker in ("double", "square", "negate"):
        g.add_edge(START, worker)              # fan out: all three start together
    g.add_edge(["double", "square", "negate"], "total")   # fan in: total waits for all three
    g.add_edge("total", END)
    return g.compile()


try:
    build(Plain).invoke({"n": 3})
except InvalidUpdateError as e:
    print("without a reducer:", type(e).__name__)

print("with a reducer:   ", build(Merged).invoke({"n": 3}))

without a reducer: InvalidUpdateError
with a reducer:    {'n': 3, 'results': {'double': 6, 'negate': -3, 'square': 9}, 'total': 12}


## 3. Conditional edges (a loop with a cap)

A **conditional edge** runs a small function after a node and uses its answer to pick the next node. Pointing the answer back at an earlier node makes a loop. The cap (`attempts >= 3`) is what stops it looping forever, like `max_rounds` in the project.

In [3]:
class Draft(TypedDict, total=False):
    text: str
    attempts: int


def improve(state: Draft) -> dict:
    return {"text": state["text"] + "!", "attempts": state.get("attempts", 0) + 1}


def good_enough(state: Draft) -> str:
    done = len(state["text"]) >= 6 or state["attempts"] >= 3
    return "stop" if done else "again"


g = StateGraph(Draft)
g.add_node("improve", improve)
g.add_edge(START, "improve")
g.add_conditional_edges("improve", good_enough, {"again": "improve", "stop": END})
loop = g.compile()

print(loop.invoke({"text": "hello"}))     # 6 characters after one pass: stops because it is long enough
print(loop.invoke({"text": ""}))          # never gets long enough: stops because of the cap (3 attempts)

{'text': 'hello!', 'attempts': 1}
{'text': '!!!', 'attempts': 3}


## 4. Interrupt and resume (a human in the loop)

`interrupt(...)` pauses the graph and hands a message to the caller. The graph stops and its state is saved. Later the caller resumes it with `Command(resume=...)` and **that value becomes the return value of `interrupt`**.

Two rules worth remembering, both from the project:
1. It needs a **checkpointer**, which is where the paused state is saved.
2. When it resumes, the node starts again **from the top**. So put `interrupt` first in the node, before anything that must not run twice.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


class Order(TypedDict, total=False):
    proposal: str
    decision: str
    result: str


def propose(state: Order) -> dict:
    return {"proposal": "refund 20 dollars"}


def gate(state: Order) -> dict:
    decision = interrupt({"please review": state["proposal"]})   # first statement: pauses here
    return {"decision": decision}


def apply(state: Order) -> dict:
    return {"result": f"applied: {state['proposal']}"}


def skip(state: Order) -> dict:
    return {"result": "nothing was done"}


g = StateGraph(Order)
for fn in (propose, gate, apply, skip):
    g.add_node(fn.__name__, fn)
g.add_edge(START, "propose")
g.add_edge("propose", "gate")
g.add_conditional_edges("gate", lambda s: "yes" if s["decision"] == "approve" else "no", {"yes": "apply", "no": "skip"})
g.add_edge("apply", END)
g.add_edge("skip", END)
review = g.compile(checkpointer=InMemorySaver())      # the checkpointer is required for interrupt

config = {"configurable": {"thread_id": "order-1"}}
first = review.invoke({}, config)
print("paused. message for the human:", first["__interrupt__"][0].value)
print("next node waiting:", review.get_state(config).next)

final = review.invoke(Command(resume="approve"), config)   # the human answers
print("after approve:", final["result"])

paused. message for the human: {'please review': 'refund 20 dollars'}
next node waiting: ('gate',)
after approve: applied: refund 20 dollars


## 5. Checkpointer and thread id

The checkpointer saves the state under a **thread id**. Same id means the same run. A different id is a separate run with its own saved state. In the project, each incident run uses its own thread id, so many runs can wait for approval at once, and rejecting one does not affect another.

In [5]:
a = {"configurable": {"thread_id": "order-A"}}
b = {"configurable": {"thread_id": "order-B"}}

review.invoke({}, a)          # both runs pause at the gate
review.invoke({}, b)
print("A is waiting at:", review.get_state(a).next)
print("B is waiting at:", review.get_state(b).next)

print("\napprove A, reject B")
print("A:", review.invoke(Command(resume="approve"), a)["result"])
print("B:", review.invoke(Command(resume="reject"), b)["result"])

print("\nthe saved state of A is still there:", review.get_state(a).values)

A is waiting at: ('gate',)
B is waiting at: ('gate',)

approve A, reject B
A: applied: refund 20 dollars
B: nothing was done

the saved state of A is still there: {'proposal': 'refund 20 dollars', 'decision': 'approve', 'result': 'applied: refund 20 dollars'}


## 6. Putting it together: a three-layer graph

Same shape as the incident-triage graph, with numbers instead of incidents.

```
                      +--> low  (1-3) --+      +--> even_odd --+
 START -> validate ---+--> mid  (4-7) --+------+--> prime    --+--> finalize -> END
             |        +--> high (8-10) -+      +--> size     --+
             +-- not a number 1-10 --> reject -> END
```

| Layer | Nodes | Like in the project | LangGraph feature |
|---|---|---|---|
| 1 | `validate` | Triage supervisor | conditional edge (valid or reject) |
| 2 | `low`, `mid`, `high` | Logs, Metrics, Change agents | fan-out, reducer on `findings` |
| 3 | `even_odd`, `prime`, `size` | a second wave of specialists | fan-in (wait for all of layer 2), fan-out again |
| 4 | `finalize` | Postmortem | fan-in, returns the final answer |

Each node follows the project's pattern: read state, say what it is doing (`trace`), do its job, return only its own piece.

In [6]:
import operator


def merge_findings(a: dict, b: dict) -> dict:       # reducer, same idea as the project's _merge
    return {**a, **b}


class Pipeline(TypedDict, total=False):
    raw: str                                              # what the user typed
    number: int                                           # set by validate
    error: str                                            # set by validate when the input is bad
    findings: Annotated[dict, merge_findings]             # layers 2 and 3 all write here
    trace: Annotated[list, operator.add]                  # every node appends a line (like em.say)
    final: int                                            # set by finalize


def say(node: str, text: str) -> dict:
    return {"trace": [f"{node:9} {text}"]}


# ---- layer 1: the supervisor
def validate(state: Pipeline) -> dict:
    raw = state["raw"].strip()
    if not raw.lstrip("-").isdigit():
        return {"error": f"'{raw}' is not a whole number", **say("validate", "rejected: not a number")}
    n = int(raw)
    if not 1 <= n <= 10:
        return {"error": f"{n} is outside 1-10", **say("validate", "rejected: out of range")}
    return {"number": n, **say("validate", f"{n} is valid, sending to the range agents")}


LAYER2 = ["low", "mid", "high"]
LAYER3 = ["even_odd", "prime", "size"]


def route_after_validate(state: Pipeline):
    return "reject" if state.get("error") else LAYER2     # a list means: run all of these in parallel


def reject(state: Pipeline) -> dict:
    return say("reject", state["error"])


# ---- layer 2: three range agents, run in parallel, each checks its own range
def range_agent(name: str, low: int, high: int):
    def node(state: Pipeline) -> dict:
        mine = low <= state["number"] <= high
        text = "my range, claiming it" if mine else "not mine"
        return {"findings": {name: mine}, **say(name, f"{low}-{high}: {text}")}
    node.__name__ = name
    return node


# ---- layer 3: three more agents, run in parallel after ALL of layer 2 finished
def even_odd(state: Pipeline) -> dict:
    kind = "even" if state["number"] % 2 == 0 else "odd"
    return {"findings": {"parity": kind}, **say("even_odd", kind)}


def prime(state: Pipeline) -> dict:
    n = state["number"]
    is_prime = n > 1 and all(n % d for d in range(2, n))
    return {"findings": {"prime": is_prime}, **say("prime", "prime" if is_prime else "not prime")}


def size(state: Pipeline) -> dict:
    n = state["number"]
    label = "small" if n <= 3 else "medium" if n <= 7 else "large"
    return {"findings": {"size": label}, **say("size", label)}


# ---- layer 4: finalize, waits for layer 3, returns the number
def finalize(state: Pipeline) -> dict:
    owner = next(k for k in ("low", "mid", "high") if state["findings"][k])
    return {"final": state["number"], **say("finalize", f"{state['number']} confirmed by '{owner}'")}


In [7]:
g = StateGraph(Pipeline)
g.add_node("validate", validate)
g.add_node("reject", reject)
g.add_node("low", range_agent("low", 1, 3))
g.add_node("mid", range_agent("mid", 4, 7))
g.add_node("high", range_agent("high", 8, 10))
for fn in (even_odd, prime, size, finalize):
    g.add_node(fn.__name__, fn)

g.add_edge(START, "validate")
g.add_conditional_edges("validate", route_after_validate, ["reject", *LAYER2])
g.add_edge("reject", END)
for agent in LAYER3:
    g.add_edge(LAYER2, agent)
g.add_edge(LAYER3, "finalize")
g.add_edge("finalize", END)
pipeline = g.compile()


def run(text: str):
    out = pipeline.invoke({"raw": text})
    print(f"input {text!r}")
    for line in out["trace"]:
        print("   ", line)
    print("    final:", out.get("final"), "| error:", out.get("error"), "\n")


run("6")          # valid: mid claims it
run("2")          # valid: low claims it
run("11")         # out of range: rejected in layer 1, layers 2 and 3 never run
run("abc")        # not a number: rejected in layer 1

input '6'
    validate  6 is valid, sending to the range agents
    high      8-10: not mine
    low       1-3: not mine
    mid       4-7: my range, claiming it
    even_odd  even
    prime     not prime
    size      medium
    finalize  6 confirmed by 'mid'
    final: 6 | error: None 

input '2'
    validate  2 is valid, sending to the range agents
    high      8-10: not mine
    low       1-3: my range, claiming it
    mid       4-7: not mine
    even_odd  even
    prime     prime
    size      small
    finalize  2 confirmed by 'low'
    final: 2 | error: None 

input '11'
    validate  rejected: out of range
    reject    11 is outside 1-10
    final: None | error: 11 is outside 1-10 

input 'abc'
    validate  rejected: not a number
    reject    'abc' is not a whole number
    final: None | error: 'abc' is not a whole number 



**What to notice**

| Observation | Why |
|---|---|
| `validate` always prints first | it is the only node after `START` |
| `low`, `mid`, `high` all run, even when two say "not mine" | fan-out runs every branch; each decides for itself |
| Layer 3 starts only after all three range agents | a list as the source of `add_edge` means "wait for all" |
| `finalize` prints last | it waits for all of layer 3 |
| Bad input skips layers 2 to 4 | the conditional edge sends it to `reject` |
| `findings` holds keys from six different nodes | the reducer merges parallel writes (section 2) |
| Order inside a layer may differ between runs | parallel nodes have no fixed order |

## Where to look next in the project

| Concept | File |
|---|---|
| State, reducer, nodes, edges, parallel, conditional edges | `backend/src/triage/graph.py` |
| Interrupt (the approval gate) | the `gate` node in `backend/src/triage/agents.py` |
| Resume and thread id | `RunManager.decide` and `RunManager.start` in `backend/src/triage/runs.py` |